# Spatial Panels

Observing the same places repeatedly lets a model control for everything about a place that does not change over time, measured or not. In `neighbayes`, a panel needs no new classes. The models from this afternoon become panel models when we give them a period column and a unit column, and `unit_effects` adds fixed or random effects for each unit.

We return to the county homicide data, now for all four census years from 1960 to 1990, and model the homicide rate per 100,000 residents (`HR`).

In [ ]:
import arviz as az
import geopandas as gpd
import pandas as pd
from libpysal.graph import Graph
from neighbayes.models import OLS, SAR, SEM

az.rcParams["stats.ci_prob"] = 0.95
fit_kwargs = dict(draws=1000, tune=1000, chains=4, random_seed=42)

## Building the panel

`neighbayes` sorts a panel by period, then by unit, and applies the graph within each period, so the graph must list the units in the same order as the sorted unit column. Sorting the counties by their FIPS code before building the graph guarantees it.

In [ ]:
nat = gpd.read_file("data/NAT/NAT.shp").sort_values("FIPSNO").reset_index(drop=True)
graph = Graph.build_contiguity(nat, rook=False).transform("r")

The file is wide, with one row per county and one column per variable and year: `HR60`, `HR70`, and so on. A panel model needs it long, with one row per county and year. `pd.wide_to_long` splits each column name into a stub, the variable, and a suffix, the year.

In [ ]:
stubs = ["HR", "RD", "PS", "UE", "DV", "MA"]
columns = ["FIPSNO"] + [f"{stub}{year}" for stub in stubs for year in ("60", "70", "80", "90")]

panel = pd.wide_to_long(
    pd.DataFrame(nat[columns]), stubnames=stubs, i="FIPSNO", j="year"
).reset_index()
panel["year"] = panel["year"] + 1900
panel.head()

In [ ]:
panel.groupby("year")["HR"].mean()

Homicide rose through the 1960s and 1970s and eased by 1990. The covariates are the same structural measures as before, now observed in every census year: resource deprivation (`RD`), population structure (`PS`), unemployment (`UE`), divorce (`DV`), and median age (`MA`).

## Fixed effects

With `unit_effects="fixed"`, the model applies the within transformation: every variable is measured as a deviation from its county's own mean over the four decades. Anything constant about a county, from its region to its history, drops out, and the coefficients come only from change within counties.

In [ ]:
formula = "HR ~ RD + PS + UE + DV + MA"
panel_kwargs = dict(data=panel, W=graph, time_col="year", unit_col="FIPSNO")

ols_fe = OLS(formula=formula, unit_effects="fixed", **panel_kwargs)
ols_fe.fit(**fit_kwargs)
ols_fe.summary(round_to=3)

The Lagrange multiplier tests have panel versions, and the decision tree works as it did for a cross-section.

In [ ]:
ols_fe.spatial_diagnostics()

In [ ]:
ols_fe.spatial_diagnostics_decision()

Both robust tests reject, and the tree, choosing between them, picks the spatial error model. With more than 12,000 observations, the tests have the power to find both kinds of dependence.

## Spatial lag and spatial error panels

In [ ]:
sar_fe = SAR(formula=formula, unit_effects="fixed", **panel_kwargs)
sar_fe.fit(**fit_kwargs)
sar_fe.summary(round_to=3)

In [ ]:
sar_fe.impacts()[["direct", "indirect", "total"]].round(3)

In [ ]:
sem_fe = SEM(formula=formula, unit_effects="fixed", **panel_kwargs)
sem_fe.fit(**fit_kwargs)
sem_fe.summary(round_to=3)

Both spatial parameters are near 0.2 and well away from zero. The diagnostics on each model point where they pointed in Chicago, toward the neighbors' covariates.

In [ ]:
{"SAR": sar_fe.spatial_diagnostics_decision(format="model"),
 "SEM": sem_fe.spatial_diagnostics_decision(format="model")}

## Random effects

Random effects replace each county's fixed intercept with a draw from a common distribution, which the model estimates. The change is one argument.

In [ ]:
sar_re = SAR(formula=formula, unit_effects="random", **panel_kwargs)
sar_re.fit(**fit_kwargs)
sar_re.summary(var_names=["~unit_effect"], round_to=3)

In [ ]:
pd.concat(
    {
        "fixed": sar_fe.summary()["mean"],
        "random": sar_re.summary(var_names=["~unit_effect"])["mean"],
    }, axis=1
).loc[["rho", "RD", "PS", "UE", "DV", "MA"]].round(3)

The estimates differ sharply. The effect of resource deprivation more than doubles, and the sign on population structure reverses. Random effects use the variation between counties as well as within them, which is why they are more precise, but they assume the county effects are unrelated to the covariates. When the two sets of estimates disagree this much, that assumption fails, which is the logic of a Hausman test, and the fixed effects are the safer reading.

## Beyond this afternoon

`neighbayes` also estimates spatial Tobit, zero-inflated and hurdle count models, origin-destination flows, and multilevel models. For spatial regimes, seemingly unrelated regressions, and GMM estimation, which earlier versions of this workshop taught, PySAL's `spreg` remains the place to go.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
Homicide rose and fell nationally over these decades. Refit the fixed-effects spatial lag model with <code>time_effects=True</code>, which adds a fixed effect for each census year. What happens to $\rho$?
</div>

In [ ]:
# %load solutions/10.py